# Local LLM benchmark analysis

Query one immutable Parquet snapshot with **DuckDB**, work with the results in **Polars**, and plot with **Plotly**. All computation stays local. Run cells in order.

Prepare data with `make benchmark-export`, then launch with `make notebook`. Set `EXPORT_DIR` below or `BENCHMARK_EXPORT_DIR` before launching. No dataset is modified. Synthetic data is excluded unless explicitly enabled.

In [ ]:
import json
import os
from pathlib import Path

import duckdb
import plotly.graph_objects as go
import polars as pl
from IPython.display import display

# Edit these parameters. Environment variables also support automated execution.
EXPORT_DIR = os.getenv("BENCHMARK_EXPORT_DIR") or None
MODELS = []
WORKLOADS = []
CONTEXT_LENGTHS = []
PHASES = ["warm_start", "benchmark"]
INCLUDE_SYNTHETIC = os.getenv("BENCHMARK_INCLUDE_SYNTHETIC", "0") == "1"
INVOCATION_ID = None  # Choose a run later for raw telemetry.

if EXPORT_DIR is None:
    repo = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
    snapshots = sorted((repo / "results" / "exports").glob("*/manifest.json"))
    if len(snapshots) != 1:
        available = (
            "\n".join(str(p.parent) for p in snapshots) or "(none; run make benchmark-export)"
        )
        raise ValueError("Set EXPORT_DIR to one completed snapshot. Available:\n" + available)
    EXPORT_DIR = snapshots[0].parent
EXPORT_DIR = Path(EXPORT_DIR).expanduser().resolve()
manifest_file = EXPORT_DIR / "manifest.json"
if not manifest_file.is_file():
    raise FileNotFoundError(
        f"Snapshot is missing or was cleaned: {EXPORT_DIR}. Export results again."
    )
manifest = json.loads(manifest_file.read_text())
if manifest.get("format") != "llm-bench-export" or manifest.get("schema_version") != "1.2":
    raise ValueError("Unsupported snapshot format/schema; export with this framework version.")
print("Snapshot:", EXPORT_DIR, "Schema:", manifest["schema_version"])

con = duckdb.connect(":memory:")
con.sql("SET TimeZone='UTC'")
for name in ("experiments", "inference_runs", "telemetry", "telemetry_summary"):
    con.read_parquet(str(EXPORT_DIR / f"{name}.parquet")).create_view(name)
con.sql("""
CREATE VIEW runs AS
SELECT r.*, e.model, e.model_digest, e.quantization, e.model_size_bytes,
       e.parameter_size, e.num_ctx, e.num_predict, e.temperature, e.seed, e.think,
       e.synthetic, e.backend_version, e.telemetry_host_role,
       e.hardware.cpu AS cpu_model, e.hardware.gpu AS gpu_model,
       e.hardware.total_ram_bytes AS total_ram_bytes
FROM inference_runs r JOIN experiments e USING (experiment_id)
""")
# Keep parameters separate from SQL, including model names and file paths.
conditions, params = [], []
if not INCLUDE_SYNTHETIC:
    conditions.append("NOT synthetic")
for column, values in (("model", MODELS), ("workload", WORKLOADS), ("num_ctx", CONTEXT_LENGTHS)):
    if values:
        conditions.append(f"{column} IN (SELECT unnest(?))")
        params.append(values)
all_runs = con.sql(
    "SELECT * FROM runs" + (" WHERE " + " AND ".join(conditions) if conditions else ""),
    params=params,
).pl()
# Register a Polars frame for subsequent DuckDB SQL. No persistent database needed.
con.register("filtered_runs", all_runs)
runs = all_runs.filter(pl.col("phase").is_in(PHASES))
con.register("measured_runs", runs)
print("Selected invocations:", all_runs.height, "Comparison invocations:", runs.height)
display(
    con.sql(
        "SELECT experiment_id, model, timestamp_utc, synthetic "
        "FROM experiments ORDER BY timestamp_utc"
    ).pl()
)

## Validate joins, sensors, and missing data

Every raw observation must reference exactly one invocation. Historical exports can have zero telemetry rows. Availability reasons and coverage are part of the data, not zeros to fill in. Date/time columns are UTC.

In [ ]:
for name in ("experiments", "inference_runs", "telemetry", "telemetry_summary"):
    print(name, con.sql(f"SELECT count(*) FROM {name}").fetchone()[0])
    display(con.sql(f"DESCRIBE {name}").pl())
assert con.sql("SELECT count(*) = count(DISTINCT invocation_id) FROM inference_runs").fetchone()[0]
for name in ("telemetry", "telemetry_summary"):
    orphans = con.sql(
        f"SELECT count(*) FROM {name} t LEFT JOIN inference_runs r "
        "USING (experiment_id, invocation_id) WHERE r.invocation_id IS NULL"
    ).fetchone()[0]
    assert orphans == 0, f"Orphan rows in {name}"
missingness = con.sql("""
SELECT metric, device, source, scope, unit, available, reason, count(*) AS observations
FROM telemetry GROUP BY ALL ORDER BY metric, device, source
""").pl()
display(missingness)
display(
    con.sql("""
SELECT metric, device, source, count(*) AS invocations,
       avg(coverage_fraction) AS mean_coverage, sum(missing_count) AS missing_count
FROM telemetry_summary GROUP BY ALL ORDER BY metric, device, source
""").pl()
)
if not all_runs.height:
    print(
        "No selected runs. Check filters; enable INCLUDE_SYNTHETIC only for illustrative fixtures."
    )

## Inference performance and variability

Each experiment/configuration/workload is summarized separately. Warm-ups and cold-start observations are excluded from these comparisons by default. Prompt throughput is partitioned by cache status. Variability describes repetitions; it is not a confidence interval.

In [ ]:
GROUPS = [
    "experiment_id",
    "model",
    "workload",
    "phase",
    "num_ctx",
    "num_predict",
    "temperature",
    "seed",
    "think",
    "quantization",
    "cpu_model",
    "gpu_model",
    "total_ram_bytes",
    "backend_version",
    "synthetic",
]
performance = (
    runs.group_by(GROUPS)
    .agg(
        pl.len().alias("repetitions"),
        pl.col("generation_tokens_per_second").mean().alias("mean_tokens_per_second"),
        pl.col("generation_tokens_per_second").median().alias("median_tokens_per_second"),
        pl.col("generation_tokens_per_second").std().alias("std_tokens_per_second"),
        pl.col("client_ttft_seconds").median().alias("median_ttft_seconds"),
        pl.col("client_duration_seconds").median().alias("median_latency_seconds"),
        pl.col("premature_completion").sum().alias("flagged_runs"),
    )
    .sort(["model", "workload"])
)
display(performance)
display(
    runs.group_by(GROUPS + ["prompt_cache_hit"]).agg(
        pl.col("prompt_tokens_per_second").median().alias("median_prompt_tokens_per_second"),
        pl.len().alias("runs"),
    )
)


def scatter(frame, x, y, title, group=("experiment_id", "model", "workload"), lines=False):
    usable = frame.drop_nulls([x, y])
    print(f"{title}: {usable.height} usable rows, {frame.height - usable.height} unavailable")
    if not usable.height:
        print("Unavailable: required measurements were not recorded.")
        return
    fig = go.Figure()
    plot_frame = frame.drop_nulls([x]) if lines else usable
    partitions = (
        plot_frame.partition_by(list(group), as_dict=True) if group else {"all": plot_frame}
    )
    for key, part in partitions.items():
        part = part.sort(x)
        fig.add_scatter(
            x=part[x].to_list(),
            y=part[y].to_list(),
            mode="lines+markers" if lines else "markers",
            name=str(key),
        )
    fig.update_layout(title=title, xaxis_title=x, yaxis_title=y)
    fig.show()


scatter(
    performance,
    "median_ttft_seconds",
    "median_tokens_per_second",
    "Throughput and TTFT by experiment/workload",
)
scatter(
    runs, "repetition", "generation_tokens_per_second", "Throughput variability across repetitions"
)

## Select one hardware series before joining

Choose exactly one **device/source** for each metric. This avoids multiplying inference rows when multiple GPUs or sensors exist. The initial choice is printed and can be changed below. Client-host telemetry is excluded from inference-host comparisons.

Power domains remain separate. Package/SoC readings on AMD APUs may include both CPU and GPU. Never add them together to estimate total energy.

In [ ]:
def hardware_metric(metric, device=None, source=None):
    options = con.sql(
        """
        SELECT DISTINCT t.device, t.source, t.scope, t.unit
        FROM telemetry_summary t JOIN measured_runs r USING (experiment_id, invocation_id)
        WHERE t.metric = ? AND t.valid_count > 0
          AND (r.telemetry_host_role IS NULL OR r.telemetry_host_role = 'inference_host')
        ORDER BY device, source
    """,
        params=[metric],
    ).pl()
    if device is not None:
        options = options.filter(pl.col("device") == device)
    if source is not None:
        options = options.filter(pl.col("source") == source)
    if not options.height:
        return con.sql(
            "SELECT r.*, CAST(NULL AS DOUBLE) AS sensor_mean, "
            "CAST(NULL AS DOUBLE) AS coverage_fraction, "
            "CAST(NULL AS DOUBLE) AS energy_joules, "
            "CAST(NULL AS DOUBLE) AS joules_per_generated_token "
            "FROM measured_runs r WHERE FALSE"
        ).pl()
    chosen = options.row(0, named=True)
    print(metric, "selected:", chosen, "available series:", options.height)
    selected = con.sql(
        """
        SELECT r.*, t.mean AS sensor_mean, t.coverage_fraction,
               t.energy_joules, t.joules_per_generated_token
        FROM measured_runs r JOIN telemetry_summary t USING (experiment_id, invocation_id)
        WHERE t.metric = ? AND t.device = ? AND t.source = ?
          AND (r.telemetry_host_role IS NULL OR r.telemetry_host_role = 'inference_host')
    """,
        params=[metric, chosen["device"], chosen["source"]],
    ).pl()
    assert selected["invocation_id"].n_unique() == selected.height
    return selected


# Set explicit device/source values here when comparing a particular sensor.
gpu = hardware_metric("gpu_utilization")
power = hardware_metric("power")
cpu = hardware_metric("cpu_utilization", device="system")
memory = hardware_metric("ram_used")
scatter(gpu, "sensor_mean", "generation_tokens_per_second", "GPU utilization (%) versus throughput")
scatter(
    power, "sensor_mean", "generation_tokens_per_second", "Selected power (W) versus throughput"
)
scatter(memory, "num_ctx", "sensor_mean", "RAM consumption (bytes) versus configured context")
cpu_gpu = gpu.select("invocation_id", pl.col("sensor_mean").alias("gpu_percent")).join(
    cpu.select("invocation_id", pl.col("sensor_mean").alias("cpu_percent")),
    on="invocation_id",
    how="inner",
)
display(cpu_gpu)
scatter(
    cpu_gpu, "cpu_percent", "gpu_percent", "CPU versus GPU utilization", group=()
) if cpu_gpu.height else print("CPU/GPU comparison unavailable")

## Energy efficiency and model size

Prefer an energy-counter series when available; otherwise select a power series. Energy is only available with complete coverage and acceptable gaps. Joules/token includes prompt processing and loading, divided by generated tokens. NULL means unavailable, including zero generated tokens.

Artifact size is the installed model file size, not parameter count, resident GPU memory, or a memory requirement. Separate configurations and quantization before drawing conclusions. Scatter plots are descriptive; no causal or regression claim is implied.

In [ ]:
energy = hardware_metric("energy")
if not energy["energy_joules"].drop_nulls().len():
    energy = power
print("Rows with complete energy:", energy["energy_joules"].drop_nulls().len())
display(
    energy.select(
        "invocation_id",
        "model",
        "workload",
        "coverage_fraction",
        "energy_joules",
        "joules_per_generated_token",
    )
)
scatter(
    energy,
    "generation_tokens_per_second",
    "joules_per_generated_token",
    "Throughput versus whole-invocation J/generated token",
)
scatter(
    runs,
    "model_size_bytes",
    "generation_tokens_per_second",
    "Model artifact bytes versus throughput",
)
# Compare phases without pooling experiments or workload/configuration settings.
phase_comparison = all_runs.group_by(GROUPS).agg(
    pl.len().alias("runs"),
    pl.col("client_duration_seconds").median().alias("median_latency_seconds"),
    pl.col("server_load_seconds").median().alias("median_load_seconds"),
    pl.col("generation_tokens_per_second").median().alias("median_tokens_per_second"),
)
display(phase_comparison)
print("Cold-start labels retain the legacy load-duration heuristic; inspect residency evidence.")

## Sustained inference and raw traces

Select an invocation and inspect original samples, including their missing-data reasons. Boundary observations can lie just outside the invocation; energy integration clips to the invocation interval. CPU/process utilization is interval-based and can include activity before the boundary.

Experiment time-series use observation UTC timestamps, retaining gaps and separating device/source series. NULLs are never filled with zeros.

In [ ]:
selected_id = INVOCATION_ID or (runs["invocation_id"][0] if runs.height else None)
if selected_id is None:
    print("No invocation selected.")
else:
    display(
        con.sql("SELECT * FROM filtered_runs WHERE invocation_id = ?", params=[selected_id]).pl()
    )
    trace = con.sql(
        """
        SELECT t.*, r.model, r.workload,
               coalesce(t.value_float, CAST(t.value_int AS DOUBLE)) AS numeric_value
        FROM telemetry t JOIN filtered_runs r USING (experiment_id, invocation_id)
        WHERE t.invocation_id = ?
          AND t.metric IN ('temperature', 'cpu_frequency', 'gpu_frequency',
                           'gpu_utilization', 'power')
        ORDER BY t.offset_seconds
    """,
        params=[selected_id],
    ).pl()
    display(trace)
    for metric in trace["metric"].unique().sort().to_list():
        scatter(
            trace.filter(pl.col("metric") == metric),
            "offset_seconds",
            "numeric_value",
            f"Invocation {selected_id}: {metric}",
            group=("device", "source", "unit"),
            lines=True,
        )
    experiment_id = (
        runs.filter(pl.col("invocation_id") == selected_id)["experiment_id"][0]
        if selected_id in runs["invocation_id"].to_list()
        else None
    )
    if experiment_id:
        sustained = con.sql(
            """
            SELECT t.timestamp_utc, t.invocation_id, t.metric, t.device, t.source, t.unit,
                   coalesce(t.value_float, CAST(t.value_int AS DOUBLE)) AS numeric_value
            FROM telemetry t WHERE t.experiment_id = ?
              AND t.metric IN ('temperature', 'cpu_frequency', 'gpu_frequency', 'power')
            ORDER BY t.timestamp_utc
        """,
            params=[experiment_id],
        ).pl()
        for metric in sustained["metric"].unique().sort().to_list():
            scatter(
                sustained.filter(pl.col("metric") == metric),
                "timestamp_utc",
                "numeric_value",
                f"Experiment {experiment_id}: {metric}",
                group=("device", "source", "unit", "invocation_id"),
                lines=True,
            )
con.close()

## Next experiments

Vary one setting at a time: context length, model/quantization, or output limit. Keep workload, host configuration, and phase consistent. Increase repetitions to inspect variability. Check coverage and competing system activity before interpreting utilization or energy differences.

To start fresh: stop the benchmark, preview with `make benchmark-clean DRY_RUN=1`, then run `make benchmark-clean`. Export the next run and select its new snapshot path before rerunning this notebook.